# Checkpoint-chaining probe

Tests the one mechanism the whole multi-session training plan rests on:
a private dataset written by one session must be **readable by the next**.

Motivation: `kaggle datasets files` and `datasets download` both return 403
on this account's own private dataset from outside Kaggle, even though
`datasets list --mine` sees it and `datasets version` writes to it happily.
That 403 is an API-surface permission. Inside a kernel the dataset is
**mounted**, not fetched over the API — so it may be entirely unaffected.
This notebook settles which.

If the mount works, checkpoint chaining is viable and the 403 is a local
inconvenience. If it does not, the multi-session plan needs redesigning
before a single GPU-hour is spent.

In [ ]:
# 1. did the private dataset mount at all?
from pathlib import Path

INPUT = Path("/kaggle/input")
print("attached:", [p.name for p in INPUT.iterdir()] if INPUT.exists() else "NONE")
for p in sorted(INPUT.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(INPUT), f"({p.stat().st_size}B)")

In [ ]:
# 2. is the checkpoint readable? This is the question.
cands = sorted(INPUT.rglob("ckpt_step*.pt"))
print("checkpoints found:", [str(c.relative_to(INPUT)) for c in cands])
assert cands, "PRIVATE DATASET DID NOT MOUNT -- chaining is not viable as designed"

content = cands[0].read_bytes()
print("read", len(content), "bytes ->", content[:60])
print("PRIVATE DATASET MOUNTS AND READS -- chaining viable")

In [ ]:
# 3. does our own latest_checkpoint() pick it up? The real chaining entry point,
#    not a hand-rolled glob.
import subprocess, sys

r = subprocess.run(
    ["git", "clone", "-q", "--branch", "feat/m0-m1-harness", "--depth", "1",
     "https://github.com/SonLamHG/pgmm.git", "/kaggle/working/repo"],
    capture_output=True, text=True,
)
assert r.returncode == 0, r.stderr
sys.path.insert(0, "/kaggle/working/repo")

from kaggle_harness.chain import latest_checkpoint

found = latest_checkpoint([cands[0].parent, Path("/kaggle/working/ckpt")])
print("latest_checkpoint ->", found)
assert found is not None, "latest_checkpoint failed to see the mounted checkpoint"
print("CHAIN ENTRY POINT OK")

In [ ]:
# 4. can a kernel WRITE a new version back? The other half of the loop.
#    A kernel that can read but not write cannot hand off to the next session.
import os, json, shutil

have_creds = Path("/kaggle/input").exists()
print("NOTE: writing a dataset version from inside a kernel needs Kaggle")
print("credentials available to the kernel (Secrets). Checking what we have:")
print("  KAGGLE_USERNAME set:", bool(os.environ.get("KAGGLE_USERNAME")))
print("  KAGGLE_KEY set     :", bool(os.environ.get("KAGGLE_KEY")))
print("  ~/.kaggle exists   :", Path.home().joinpath(".kaggle").exists())
print()
print("If all false, the push-back half needs a Kaggle Secret holding an API")
print("token, and that is a setup step to do before any long run.")